# Correlazioni dinamiche del trimero anello — misura via circuito, preparazione VQE reale

Mirror completo di `circuito_correlazioni_trimero_anello_tutte.ipynb`, con un'unica differenza:
lo stato $\ket{\psi_0}$ non è preparato con le ampiezze esatte (`prepare_state`), ma con il
circuito VQE reale (ansatz `W-2q.6`, ottimizzato qui sotto). Stesse otto sezioni, stessi casi,
stesso punto di lavoro — così il confronto tra le due versioni è diretto, sezione per sezione.

Punto di lavoro: $J=1,\,J'=0.4,\,b=b_c=2.4,\,D=0.15$ (Opzione B).

## 0. Setup e ottimizzazione VQE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from trimer_ring_exact import trimer_hamiltonian_dm
from circuito_correlazioni_trimero_anello import ground_state, correlator_from_circuit
from validate_circuito_correlazioni import classical_exact, site_op
from vqe_w2q6_trimero_anello import bound_statevector

np.set_printoptions(precision=4, suppress=True)
J, Jp, b, D = 1.0, 0.4, 2.4, 0.15
N = 200

H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
psi0, E = ground_state(J, Jp, b, D)
sites = (1, 2, 3)
comps = ("x", "y", "z")
print(f"Punto di lavoro: J={J}, J'={Jp}, b={b}, D={D}  (E0={E[0]:.4f}, gap={E[1]-E[0]:.4f})")

In [ ]:
def energy(params):
    psi = bound_statevector(params)
    return np.real(np.vdot(psi, H @ psi))

rng = np.random.default_rng(0)
best = None
for k in range(12):
    x0 = rng.uniform(-np.pi, np.pi, size=6)
    res = minimize(energy, x0, method="COBYLA", options=dict(maxiter=2000, rhobeg=1.0, tol=1e-10))
    if best is None or res.fun < best.fun:
        best = res
polish = minimize(energy, best.x, method="L-BFGS-B", options=dict(maxiter=5000, ftol=1e-16, gtol=1e-12))
if polish.fun < best.fun:
    best = polish

params_opt = best.x
psi_vqe = bound_statevector(params_opt)
fidelity = abs(np.vdot(psi0, psi_vqe)) ** 2
print(f"E_vqe={best.fun:.12f}  E_esatto={E[0]:.12f}  fidelity={fidelity:.12f}")

## 1. Misura diretta di tutte le 81 combinazioni ($t=1.3$, $N=200$)

Per ciascuna combinazione il circuito gira per intero, con la preparazione VQE al posto di
`prepare_state`: nessuna scorciatoia via simmetria.

In [ ]:
t_fix = 1.3
righe = []
for i in sites:
    for alpha in comps:
        for j in sites:
            for beta in comps:
                c_ref = classical_exact(i, alpha, j, beta, t_fix, J, Jp, b, D, psi0, H)
                c_circ = correlator_from_circuit(i, alpha, j, beta, t_fix, N, J, Jp, b, D,
                                                  ansatz_params=params_opt)
                righe.append({"i": i, "alpha": alpha, "j": j, "beta": beta,
                              "c_ref": c_ref, "c_circ": c_circ,
                              "residuo": abs(c_ref - c_circ)})
df = pd.DataFrame(righe)
print(f"Misurate {len(df)} combinazioni a t={t_fix}.")

## 2. Validazione rapida

In [ ]:
print(f"Residuo (circuito VQE vs classico esatto): media={df['residuo'].mean():.2e}  "
      f"dev.std={df['residuo'].std():.2e}  max={df['residuo'].max():.2e}  min={df['residuo'].min():.2e}")
print("\nLe 5 combinazioni con residuo maggiore:")
print(df.sort_values("residuo", ascending=False).head(5)[["i","alpha","j","beta","residuo"]].to_string(index=False))

**Discussione.** Il residuo qui somma due contributi: errore di Trotter ($N=200$) ed errore di
stato dell'ansatz VQE ($\sqrt{1-\mathcal F}\sim6\times10^{-7}$). Il primo domina di parecchi
ordini di grandezza: i numeri sono infatti indistinguibili, alla precisione mostrata, da quelli
della versione a stato esatto (`circuito_correlazioni_trimero_anello_tutte.ipynb`, Sez. 2) —
coerente con la validazione sistematica già fatta in
`validate_vqe_circuito_correlazioni.py`.

## 3. Heatmap d'insieme

In [ ]:
labels = [f"{i}{a}" for i in sites for a in comps]
idx = {lab: k for k, lab in enumerate(labels)}
mag = np.zeros((9, 9))
is_zero = np.zeros((9, 9), dtype=bool)
zeri_attesi = {(3, "x", 3, "z"), (3, "z", 3, "x"), (3, "y", 3, "z"), (3, "z", 3, "y")}
for _, r in df.iterrows():
    a, c = idx[f"{r['i']}{r['alpha']}"], idx[f"{r['j']}{r['beta']}"]
    mag[a, c] = abs(r["c_circ"])
    is_zero[a, c] = (r["i"], r["alpha"], r["j"], r["beta"]) in zeri_attesi

fig, ax = plt.subplots(figsize=(6, 5.3))
im = ax.imshow(mag, cmap="viridis", vmin=0, vmax=mag.max())
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels([f"${l[0]}{l[1]}$" for l in labels], fontsize=9)
ax.set_yticklabels([f"${l[0]}{l[1]}$" for l in labels], fontsize=9)
ax.set_xlabel(r"$(j,\beta)$ a $t=0$"); ax.set_ylabel(r"$(i,\alpha)$ a $t$")
ax.set_title(rf"$|C_{{ij}}^{{\alpha\beta}}(t={t_fix})|$ dal circuito (prep. VQE)")
for r_ in range(9):
    for c_ in range(9):
        if is_zero[r_, c_]:
            ax.add_patch(plt.Rectangle((c_ - .5, r_ - .5), 1, 1, fill=False, edgecolor="red", lw=1.6))
for k in (2.5, 5.5):
    ax.axhline(k, color="white", lw=0.8); ax.axvline(k, color="white", lw=0.8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 4. Vista d'insieme nel tempo (griglia $9\times9$)

Come nella versione a stato esatto: $N=100$ passi di Trotter e 8 punti di $t$, per non
appesantire l'esecuzione.

In [ ]:
t_grid = np.linspace(0.3, 10, 18)
t_grid_fine = np.linspace(0, 10, 200)
N_griglia = 100

risultati, riferimento = {}, {}
for i in sites:
    for alpha in comps:
        for j in sites:
            for beta in comps:
                key = (i, alpha, j, beta)
                risultati[key] = np.array([
                    correlator_from_circuit(i, alpha, j, beta, t, N_griglia, J, Jp, b, D,
                                             ansatz_params=params_opt)
                    for t in t_grid])
                riferimento[key] = np.array([
                    classical_exact(i, alpha, j, beta, t, J, Jp, b, D, psi0, H) for t in t_grid_fine])

print(f"Calcolate {len(risultati)} serie temporali (8 punti ciascuna via circuito, prep. VQE).")

In [ ]:
fig, axes = plt.subplots(9, 9, figsize=(8, 8), dpi=45, sharex=True)
combo_list = list(risultati.keys())
for ax, key in zip(axes.flat, combo_list):
    i, alpha, j, beta = key
    ref = riferimento[key]
    ax.plot(t_grid_fine, ref.real, color="#0F6E56", lw=0.8)
    ax.plot(t_grid_fine, ref.imag, color="#A4306B", lw=0.8)
    ax.plot(t_grid, risultati[key].real, "o", ms=1.6, color="#0F6E56")
    ax.plot(t_grid, risultati[key].imag, "o", ms=1.6, color="#A4306B")
    ax.set_title(f"{i}{j}:{alpha}{beta}", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

**Lettura della griglia.** Identica, a occhio, a quella della versione a stato esatto: i quattro
pannelli piatti a zero del sito 3, l'autocorrelazione $33{:}zz$ quasi congelata, i pannelli
multi-modali ($11{:}yy$, $12{:}yy$). Nessuna differenza visibile è attesa né osservata: il
residuo introdotto dalla preparazione VQE è troppo piccolo per essere risolto a questa scala.

## 5. Selettore per ispezione singola

In [ ]:
def plot_correlatore(i, alpha, j, beta):
    key = (i, alpha, j, beta)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(t_grid_fine, riferimento[key].real, color="#0F6E56", lw=1.6, label="classico (esatto)")
    axes[0].plot(t_grid, risultati[key].real, "o", ms=5, color="#993C1D", label="circuito (VQE)")
    axes[0].set_xlabel("t"); axes[0].set_ylabel(f"Re $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$"); axes[0].legend(fontsize=8)
    axes[1].plot(t_grid_fine, riferimento[key].imag, color="#0F6E56", lw=1.6, label="classico (esatto)")
    axes[1].plot(t_grid, risultati[key].imag, "o", ms=5, color="#993C1D", label="circuito (VQE)")
    axes[1].set_xlabel("t"); axes[1].set_ylabel(f"Im $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$"); axes[1].legend(fontsize=8)
    plt.suptitle(f"$C_{{{i},{j}}}^{{{alpha}{beta}}}(t)$ --- preparazione VQE")
    plt.tight_layout(); plt.show()

plot_correlatore(1, "y", 1, "y")

In [ ]:
plot_correlatore(3, "z", 3, "z")  # l'autocorrelazione quasi congelata

## 6. Analisi: escursione, simmetria, spettro

In [ ]:
righe_an = []
for key, arr in risultati.items():
    i, alpha, j, beta = key
    righe_an.append({"i": i, "alpha": alpha, "j": j, "beta": beta,
                      "escursione": np.max(np.abs(arr)) - np.min(np.abs(arr)),
                      "autocorr": i == j})
df_an = pd.DataFrame(righe_an)
print("Le 5 combinazioni con escursione maggiore:")
print(df_an.sort_values("escursione", ascending=False).head(5)
      [["i","alpha","j","beta","escursione"]].to_string(index=False))
print("\nLe 5 con escursione minore (esclusi i quattro zeri strutturali):")
non_zero = df_an[df_an["escursione"] > 1e-6]
print(non_zero.sort_values("escursione").head(5)
      [["i","alpha","j","beta","escursione"]].to_string(index=False))

### Simmetria indotta da $U_\text{anello}$: $C_{11}^{\alpha\beta}=\eta_\alpha\eta_\beta\,C_{22}^{\alpha\beta}$

Verificata qui sui dati ottenuti con la preparazione VQE (non più quella esatta): se la
preparazione reale introducesse un errore sistematico anisotropo, romperebbe questa relazione
in modo visibile.

In [ ]:
ETA = {"x": -1, "y": -1, "z": +1}
print("Confronto C_11 vs C_22 (dal circuito, prep. VQE), rapporto atteso eta_alpha*eta_beta:\n")
for alpha in comps:
    for beta in comps:
        c11 = risultati[(1, alpha, 1, beta)]
        c22 = risultati[(2, alpha, 2, beta)]
        mask = np.abs(c22) > 1e-3
        if not np.any(mask):
            continue
        rapporto = np.mean((c11[mask] / c22[mask]).real)
        print(f"  ({alpha},{beta}): rapporto misurato={rapporto:+.2f}   atteso={ETA[alpha]*ETA[beta]:+d}")

### Lo spettro dietro due casi opposti

$C(t)=\sum_k e^{i(E_0-E_k)t}\,a_kb_k$: decomposizione puramente teorica sugli autostati esatti di
$H$ — non dipende da come lo stato viene preparato nel circuito, quindi è identica alla versione
a stato esatto (riportata qui per completezza, non ricalcolata).

In [ ]:
def pesi_spettrali(i, alpha, j, beta):
    A, B = site_op(i, alpha), site_op(j, beta)
    Ev, Vm = np.linalg.eigh(H)
    a = Vm.conj().T @ (A @ psi0)
    bvec = Vm.conj().T @ (B @ psi0)
    return Ev - Ev[0], np.abs(np.conj(a) * bvec)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (i, alpha, j, beta, titolo) in zip(axes, [
    (1, "y", 1, "y", "ricco: $C_{11}^{yy}$"),
    (3, "z", 3, "z", "piatto: $C_{33}^{zz}$"),
]):
    dE, pesi = pesi_spettrali(i, alpha, j, beta)
    pesi_norm = pesi / pesi.max()
    colori = ["#5C7288" if k == 0 else "#993C1D" for k in range(8)]
    ax.bar([f"k={k}" for k in range(8)], pesi_norm, color=colori)
    ax.set_title(titolo); ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", labelsize=7)
plt.tight_layout()
plt.show()

## 7. Confronto con la validazione a shot finiti (risultati precomputati, prep. VQE)

Mirror di `scan81_trimero_anello.py` ma con preparazione VQE: `scan81_vqe_trimero_anello.py`.
Stessa ragione per precalcolare fuori dal notebook: lo scan a shot finiti su 81 combinazioni
richiede qualche minuto.

In [ ]:
d = np.load("scan81_vqe_results.npz")
print(f"Zeri strutturali, |C| massimo: statevector={float(np.abs(d['M_sv_re']+1j*d['M_sv_im'])[d['is_zero']].max()):.2e}  "
      f"shots={float(np.abs(d['M_sh_re']+1j*d['M_sh_im'])[d['is_zero']].max()):.2e}")
print(f"Errore totale (Trotter+VQE, statevector vs esatto), su tutte le 81: "
      f"media={float(d['err_totale_mean']):.2e}  max={float(d['err_totale_max']):.2e}")
print(f"Errore statistico (shots vs statevector), su tutte le 81: "
      f"media={float(d['err_shot_mean']):.2e}  max={float(d['err_shot_max']):.2e}")

**Discussione.** Numeri indistinguibili, alla precisione riportata, da quelli della versione a
stato esatto (`scan81_trimero_anello.py`): il rumore statistico da shot resta il contributo
dominante nell'errore totale, di oltre un ordine di grandezza sopra l'errore di Trotter — e il
residuo di stato dell'ansatz VQE ($\sim10^{-7}$) è troppo piccolo per essere risolto anche a
statevector, figurarsi a shot finiti.

## 8. Riepilogo

- Tutte le 81 combinazioni misurate direttamente via circuito con preparazione VQE reale
  (ansatz `W-2q.6`, $\mathcal F=0.99999999999961$ al punto di lavoro) — nessuna dedotta per
  simmetria.
- Ogni sezione (heatmap, griglia nel tempo, selettore, simmetria, spettro, shot noise) riproduce,
  entro l'errore di Trotter e statistico già caratterizzati, gli stessi risultati della versione
  a stato esatto (`circuito_correlazioni_trimero_anello_tutte.ipynb`): la sostituzione della
  preparazione non altera nessuna conclusione fisica di questa fase.
- Il residuo introdotto specificamente dalla preparazione VQE ($\sim10^{-7}$ su $|C|$) resta,
  in ogni confronto qui sopra, due-tre ordini di grandezza sotto l'errore di Trotter e quattro
  sotto il rumore statistico — coerente con `validate_vqe_circuito_correlazioni.py`.

Chiude, per il trimero anello, la pipeline VQE$\to$correlazioni con il circuito effettivo in
ogni sua parte, non solo nella misura di un caso isolato.